# Privacy Lens · YOLO 학습 노트북

팀 공유 드라이브 폴더(`PrivacyLens-YOLO`)에 모인 사진으로 YOLO 를 학습합니다.
**처음이면 위에서부터 차례로**, 익숙하면 필요한 칸만 실행하세요. 자세한 설명은 `vision/README.md`.

| 하고 싶은 일 | 실행할 칸 |
|---|---|
| 새 사진에 초벌 라벨 붙이기 | 0 → 1 → 2 |
| 검수 끝난 사진으로 학습하기 | 0 → 1 → 3 → 4 |
| 현황만 보기 | 0 → 1 → 5 |

GPU 사용: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**. (초벌 라벨·현황은 CPU 로도 됩니다)

## 0. 설정 — 이름과 폴더 위치

In [ ]:
WHO = ""          # 내 이름 (학습 기록에 남습니다)
ROOT = "/content/drive/MyDrive/PrivacyLens-YOLO"   # 공유 폴더 '바로가기'를 내 드라이브 맨 위에 추가했을 때의 경로
BRANCH = "main"   # vision/ 이 아직 main 에 합쳐지기 전이면 그 브랜치 이름으로

## 1. 드라이브 연결 · 코드 받기 · 설치 (세션마다 한 번)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
assert os.path.isdir(ROOT), f"{ROOT} 가 없습니다. 공유 폴더를 '내 드라이브에 바로가기 추가' 했는지 확인하세요."

if not os.path.isdir("/content/privacy-lens"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/2026-usw-project/privacy-lens", "/content/privacy-lens"], check=True)
%cd /content/privacy-lens
!git pull -q
!pip -q install ultralytics qrcode pillow-heif opencv-python-headless
os.environ["PL_YOLO_ROOT"] = ROOT
!python -m vision init

## 2. 새 사진 받기 + 초벌 라벨

`inbox/` 에 들어온 사진의 위치정보(EXIF)를 지우고 `review/` 로 옮깁니다. 학습된 모델이 있으면 박스를 미리 그려 둡니다.
끝나면 각자 컴퓨터에서 공유 폴더의 **`라벨검수도구.html`** 을 크롬으로 열어 검수하세요.

In [ ]:
!python -m vision ingest --who "$WHO"

## 3. 검수 끝난 사진을 데이터셋으로

In [ ]:
!python -m vision build --who "$WHO"

## 4. 학습

지금 모델에서 이어서 학습하고, **같은 시험 사진**으로 기존 모델과 비교해 더 좋거나 같을 때만 교체합니다.
T4 기준 사진 수백 장, 60 epoch 에 수십 분 걸립니다. 세션이 끊기면 이 칸만 다시 실행하면 됩니다.

In [ ]:
EPOCHS = 60
!python -m vision train --who "$WHO" --epochs $EPOCHS

In [ ]:
# 학습 곡선과 혼동 행렬
from IPython.display import Image, display
ver = open(f"{ROOT}/models/current.txt").read().strip()
for name in ("results.png", "confusion_matrix.png"):
    p = f"{ROOT}/models/{ver}/{name}"
    if os.path.exists(p):
        display(Image(p, width=900))

## 5. 현황

In [ ]:
!python -m vision status
import pandas as pd
h = f"{ROOT}/models/history.csv"
pd.read_csv(h, encoding="utf-8-sig") if os.path.exists(h) else "아직 학습 기록이 없습니다."

## (처음 한 번만) 합성 사진으로 첫 모델 만들기

실사가 거의 없을 때 초벌 라벨을 시작하려고 합성 송장·학생증을 학습에 넣습니다(학습 전용, 평가에는 안 씀).
실행한 뒤 4번 칸으로 학습하세요.

In [ ]:
!python -m vision synth --n 300 --who "$WHO"